# Transfer exact saved compact proof JSON through immutable CPU log
Twelve JSON artifacts across nine shards; no training, scoring or Final.


In [ ]:
import base64
import hashlib
import json
import pathlib
import zlib

SOURCES = {
    "shard_0001-compact-proof.json": (
        "hung-compact-proof-shard0001-20261028-v1",
        "compact-proof.json",
        21399,
        "75862528c5d56a27fd5c164c616faf69bcff5c7b23451e09c20ffcdae2cd87e6",
    ),
    "shard_0002-compact-proof.json": (
        "hung-compact-proof-group02-20261028-v1",
        "shard_0002-compact-proof.json",
        21388,
        "735a225b2c6253b3357c325ae09735110fe39956c17e0dbbec037e064218a99b",
    ),
    "shard_0003-compact-proof.json": (
        "hung-compact-proof-group02-20261028-v1",
        "shard_0003-compact-proof.json",
        21346,
        "e23351b9593f1703a075e9fb2ddc935d35f52c43469dbd1b3c59db32fab7e5d8",
    ),
    "shard_0008-compact-proof.json": (
        "hung-compact-proof-group03-20261028-v1",
        "shard_0008-compact-proof.json",
        25086,
        "460ee5c8703ca6a35f7d4e1a06e038d93b5d8eecd5c3f8df7ad583ac0fcfbd38",
    ),
    "shard_0009-compact-proof.json": (
        "hung-compact-proof-group03-20261028-v1",
        "shard_0009-compact-proof.json",
        25098,
        "509f30f21455a65579a7a2d8e0083143747989495295d36d5f3272ca45f23f51",
    ),
    "shard_0010-compact-proof.json": (
        "hung-compact-proof-group03-20261028-v1",
        "shard_0010-compact-proof.json",
        25044,
        "6f03412c5386dd65d7c99bf8e7d67e2d9fa3a85b82454749146c48bf725b3a3f",
    ),
    "shard_0011-compact-proof.json": (
        "hung-compact-proof-group03-20261028-v1",
        "shard_0011-compact-proof.json",
        22791,
        "632d6531c2d53644a9a1ab7209f04c2067b9e0e6b76973cf3268b9bb0204309e",
    ),
    "shard_0012-compact-proof.json": (
        "hung-compact-proof-group03-20261028-v1",
        "shard_0012-compact-proof.json",
        21584,
        "dcea62992dd8c65824739b69571727ad68db33a97d7022c8fde55c08c747ba9b",
    ),
    "shard_0013-compact-proof.json": (
        "hung-compact-proof-group03-20261028-v1",
        "shard_0013-compact-proof.json",
        21569,
        "b43feb6603fd6f09f5202411da643c8fa306b7ae4c999edef3cce5e624b01a36",
    ),
    "shard_0001-readback.json": (
        "hung-compact-proof-shard0001-readback-20261028-v1",
        "hung-compact-proof-shard0001-readback-20261028-v1.json",
        None,
        None,
    ),
    "group02-readback.json": (
        "hung-compact-proof-group02-readback-20261028-v1",
        "hung-compact-proof-group02-readback-20261028-v1.json",
        None,
        None,
    ),
    "group03-readback.json": (
        "hung-compact-proof-group03-readback-20261028-v1",
        "hung-compact-proof-group03-readback-20261028-v1.json",
        None,
        None,
    ),
}
VERSION_IDS = {
    "hung-compact-proof-shard0001-20261028-v1": 363099687,
    "hung-compact-proof-group02-20261028-v1": 363105577,
    "hung-compact-proof-shard0001-readback-20261028-v1": 363102461,
    "hung-compact-proof-group02-readback-20261028-v1": 363107432,
    "hung-compact-proof-group03-20261028-v1": 363111778,
    "hung-compact-proof-group03-readback-20261028-v1": 363113838,
}
root = pathlib.Path("/kaggle/input")
file_manifest = []
for name, (notebook, filename, expected_bytes, expected_sha) in SOURCES.items():
    matches = [p for p in root.rglob(filename) if notebook in str(p)]
    assert len(matches) == 1, (name, [str(p) for p in matches])
    raw = matches[0].read_bytes()
    sha = hashlib.sha256(raw).hexdigest()
    if expected_bytes is not None:
        assert len(raw) == expected_bytes and sha == expected_sha
    value = json.loads(raw)
    if name.endswith("compact-proof.json"):
        assert value["schema"] == "hung-kaggle-compact-shard-proof-v1"
        assert len(value["rows"]) == 20
        assert all(row["status"] == "VERIFIED_ARCHIVE_BYTES" for row in value["rows"])
    else:
        assert value["status"] == "REMOTE_PROOF_READBACK_VERIFIED"
    encoded = base64.b64encode(zlib.compress(raw, 9)).decode("ascii")
    chunks = [encoded[i : i + 3000] for i in range(0, len(encoded), 3000)]
    file_manifest.append(
        {
            "name": name,
            "notebook": notebook,
            "claimed_source_version_id": VERSION_IDS[notebook],
            "bytes": len(raw),
            "sha256": sha,
            "chunks": len(chunks),
        }
    )
    for index, data in enumerate(chunks):
        print(
            json.dumps(
                {"compact_transport_chunk": name, "index": index, "data": data}, sort_keys=True
            ),
            flush=True,
        )
print(json.dumps({"compact_transport_manifest": file_manifest}, sort_keys=True), flush=True)